# SIF Data Exploration

This notebook will guide you through the steps involved in collecting solar-induced fluorescence (SIF) data from NASA's Goddard Earth Sciences Data and Information Services Center (GES DISC), an online archive that stores data from the Orbiting Carbon Observatory-3 (OCO-3) spacecraft, among others.

The first code block below will simply import some necessary helper functions for exploring and displaying the data.

In [1]:
import calendar
from datetime import datetime
from dotenv import load_dotenv
import earthaccess
from IPython.display import display, Markdown
from netCDF4 import Dataset
import numpy as np
import os
import sys
import textwrap

# Add src directory containing helper code to sys.path
sys.path.append(os.path.abspath("../src"))

from pysif import granule_date, plot_samples, plot_gridded, create_gridded_raster

# an additional helper function for displaying long lists
def wrapped_markdown_list(my_list, width=160):
    wrapped_text = textwrap.fill(", ".join(my_list), width=width)
    display(Markdown(f"```\n{wrapped_text}\n```"))

# netCDF files can organize variables into groups, e.g. "Metadata/MeasurementMode". This helper
# lists the full path of every variable in a granule, including those inside groups.
def list_variables(group, prefix=""):
    names = [prefix + name for name in group.variables]
    for name, subgroup in group.groups.items():
        names += list_variables(subgroup, prefix + name + "/")
    return names

## I. Getting granules from GES DISC

The GES DISC stores various datasets associated with the OCO-2 and OCO-3 instruments, and in this training we will be focusing on the "SIF Lite" datasets as they have already received L2 processing to extract chlorophyll fluorescence signatures in the $O_2$ A-band at 757 and 771 nm. These datasets are hosted in NASA's Earthdata Cloud, and we will search for and download them using [earthaccess](https://earthaccess.readthedocs.io/), a Python library that handles logging in with your Earthdata Login account, searching NASA's Common Metadata Repository (CMR), and downloading data. Each dataset (also called a "collection") is identified by a short name and a version; in this exercise we will use the `OCO3_L2_Lite_SIF` dataset, version `11r`. You can also browse the available datasets and their granules in your web browser using Earthdata Search: [https://search.earthdata.nasa.gov/search?q=OCO3_L2_Lite_SIF](https://search.earthdata.nasa.gov/search?q=OCO3_L2_Lite_SIF)

A "granule" is an instrument data file, typically in netCDF (.nc or .nc4) format, containing a set of related variables from a time range of observations. Granules in general can be daily or subdaily in time cadence. In the case of the OCO3_L2_Lite_SIF.11r dataset we are are looking at today, individual netCDFs on GES DISC correspond to a single day worth of instrument observations. 

The cell below logs you in to NASA Earthdata using the `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` that you put in your `.env` file (see the README). If earthaccess can't find these, it will ask you to type in your username and password instead. Each SIF Lite granule is only 10 - 20 MB, so after logging in we will download a single granule to explore the variables it contains. These variables let us know what ancillary data is available alongside the main SIF retrievals. Later on we will use some of these ancillary variables for filtering and post-processing the data.

**If you want to look at a different dataset, change the values of the `short_name` and `version` variables and re-run this cell. You can use the Earthdata Search link earlier in this section to find other valid dataset names.**

In [2]:
# Log in to NASA Earthdata. load_dotenv() reads your username and password from the .env
# file in the repository root, and earthaccess will prompt you for them if they are not found.
load_dotenv()
auth = earthaccess.login()

short_name = "OCO3_L2_Lite_SIF"
version = "11r" # See Section V for an example with the OCO2_L2_Lite_SIF dataset
# The dataset name used by GES DISC, which we will also use to organize downloaded files
dataset = f"{short_name}.{version}"

print(f"Getting time range for {dataset} data...")
collection = earthaccess.search_datasets(short_name=short_name, version=version)[0]
time_range = collection["umm"]["TemporalExtents"][0]["RangeDateTimes"][0]
start = time_range["BeginningDateTime"][:10]
# Datasets that are still being produced do not have an end date
end = time_range.get("EndingDateTime", "present")[:10]
print(f"{dataset} has time range {start} to {end}")

Getting time range for OCO3_L2_Lite_SIF.11r data...
OCO3_L2_Lite_SIF.11r has time range 2019-08-06 to present


**If you want to look at a different date, change the date specified in `data_date` and re-run the cell below!**

In [3]:
# Get the OCO-3 SIF Lite V11r product from December 1, 2019
data_date = datetime(2019, 12, 1) # Replace with a different date if you'd like

# Search CMR for the granule from this date.
# Passing True as the third value of temporal excludes granules that only touch the edge
# of the date range, like the previous day's granule, which ends at midnight on the start date
results = earthaccess.search_data(
    short_name=short_name,
    version=version,
    temporal=(data_date.date(), data_date.date(), True),
)
if not results:
    raise FileNotFoundError(f"No {dataset} granule found for {data_date.strftime('%Y-%m-%d')}")

# Download the granule and open it. Files you have already downloaded are not downloaded again.
granule_files = earthaccess.download(results, local_path=f"data/{dataset}")
granule = Dataset(granule_files[0], "r")
print(f"\n\nThe {dataset} granule from {data_date.strftime('%d/%m/%Y')} has the following variables:")

wrapped_markdown_list(list_variables(granule))

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]



The OCO3_L2_Lite_SIF.11r granule from 01/12/2019 has the following variables:


```
Delta_Time, SZA, VZA, SAz, VAz, Longitude, Latitude, Longitude_Corners, Latitude_Corners, SIF_740nm, SIF_Uncertainty_740nm, Daily_SIF_740nm, Daily_SIF_757nm,
Daily_SIF_771nm, Quality_Flag, SimplyGoodOrBadQualityFlag, Cloud/surface_albedo_abp, Cloud/cloud_flag_abp, Cloud/delta_pressure_abp, Cloud/co2_ratio,
Cloud/o2_ratio, Diagnostics/detailed_quality_bitflag_best, Diagnostics/detailed_quality_bitflag_good, Diagnostics/quality_bitflag_thresholds_best,
Diagnostics/quality_bitflag_thresholds_good, Geolocation/time_tai93, Geolocation/solar_zenith_angle, Geolocation/solar_azimuth_angle,
Geolocation/sensor_zenith_angle, Geolocation/sensor_azimuth_angle, Geolocation/altitude, Geolocation/longitude, Geolocation/latitude,
Geolocation/footprint_longitude_vertices, Geolocation/footprint_latitude_vertices, Metadata/CollectionLabel, Metadata/BuildId, Metadata/OrbitId,
Metadata/SoundingId, Metadata/FootprintId, Metadata/MeasurementMode, Meteo/surface_pressure, Meteo/specific_humidity, Meteo/vapor_pressure_deficit,
Meteo/temperature_skin, Meteo/temperature_two_meter, Meteo/wind_speed, Offset/signal_histogram_bins, Offset/signal_histogram_757nm,
Offset/signal_histogram_771nm, Offset/SIF_Relative_Mean_757nm, Offset/SIF_Mean_757nm, Offset/SIF_Relative_Median_757nm, Offset/SIF_Median_757nm,
Offset/SIF_Relative_SDev_757nm, Offset/SIF_Relative_Mean_771nm, Offset/SIF_Mean_771nm, Offset/SIF_Relative_Median_771nm, Offset/SIF_Median_771nm,
Offset/SIF_Relative_SDev_771nm, Science/sounding_l1b_quality_flag, Science/IGBP_index, Science/continuum_radiance_757nm, Science/SIF_757nm,
Science/SIF_Unadjusted_757nm, Science/SIF_Relative_757nm, Science/SIF_Unadjusted_Relative_757nm, Science/SIF_Uncertainty_757nm,
Science/continuum_radiance_771nm, Science/SIF_771nm, Science/SIF_Unadjusted_771nm, Science/SIF_Relative_771nm, Science/SIF_Unadjusted_Relative_771nm,
Science/SIF_Uncertainty_771nm, Science/daily_correction_factor, Science/sounding_land_fraction, Sequences/SequencesName, Sequences/SequencesId,
Sequences/SequencesMode, Sequences/SequencesIndex, Sequences/SegmentsIndex
```

## II. Download Variables and Plot

As mentioned previously, the SIF granules contain different many different variables needed for further analysis. To get a quick sense of where observations were acquired on this particular day, we can read the Latitude and Longitude coordinates alongside the Daily_SIF_757nm variable from the granule. The value for SIF is colormapped using the viridis colormap by default.

The first time you run this code block, you will get a few warnings from cartopy notifying you that it is downloading public resources for displaying the map context, this is expected and not a problem. It may take 20-30 seconds to download all the data, so please be patient.

Note that some of the SIF samples have negative values, this is normal and expected.

In [ ]:
def get_variable_array(variable: str):
    return np.array(granule[variable][:])

# Note: For some datasets, these variables may be called "latitude" or "longitude"
lat = get_variable_array("Latitude")
lon = get_variable_array("Longitude")
sif = get_variable_array("Daily_SIF_757nm")
# Setting vmax to 0.8 W/m^2/sr/μm improves the contrast of the colormapped samples
# and is based on a priori knowledge of the data range in this granule
# Remove the vmax keyword if you want matplotlib to set the data range automatically
plot_samples(
    sif, lat, lon,
    vmin=-0.2,
    vmax=0.8,
    cmap="YlGn",
    title=f"757nm SIF ({data_date.strftime('%Y-%m-%d')})",
    label=r"SIF (W/$\mathrm{m}^2$/sr/μm)"
)

## III. Download a set of Data (Optional)

Now we can download a set of granules across a date range to perform analysis. The code will print the number of granules it found and their total size before downloading them. The following cell will download one month of data, but you can try different time ranges.

**Troubleshooting**: Some file downloads may fail. Simply retrying the operation will only download the files that you do not already have. If downloads keep failing, you can add `threads=2` to the `earthaccess.download` call so that it makes fewer requests at the same time.

In [ ]:
# Simply put in the year and month you are interested in and the timerange will be
# downloaded into a named directory for you (e.g., 2020-06)
year = 2020
month = 6
start_date = datetime(year, month, 1)
_, num_days = calendar.monthrange(year, month)
end_date = datetime(year, month, num_days)

# Passing True as the third value of temporal excludes granules that only touch the edge
# of the date range, like the previous day's granule, which ends at midnight on the start date
results = earthaccess.search_data(
    short_name=short_name,
    version=version,
    temporal=(start_date.date(), end_date.date(), True),
)
# Each search result stores the size of its granule in MB
total_mb = sum(result["size"] for result in results)
print(f"Found {len(results)} granules ({total_mb:.0f} MB)")

dl_granules = earthaccess.download(results, local_path=f"data/{dataset}/{year}-{month:02d}")

if dl_granules:
    print(f"Download summary: Downloaded {len(dl_granules)} out of {num_days} days")
    for granule_file in sorted(dl_granules):
        print(granule_file.name)

# Days with no granule, for example when the instrument was not collecting data
found_dates = [granule_date(result) for result in results]
all_dates = [datetime(year, month, day) for day in range(1, num_days + 1)]
dates_notfound = [dt for dt in all_dates if dt not in found_dates]
if dates_notfound:
    print(f"No data found for {len(dates_notfound)} days:")
    for dt in dates_notfound:
        print(dt.strftime("%d/%m/%Y"))

## IV. Generate a Mean Daily SIF Product for One Month (PLEASE READ)

**Now let's create a gridded mean daily SIF raster from the granules for one month. There are two options for sourcing your data, you only have to run one of these cells:**
- **Option A:** Use the locally downloaded files from Section III.
- **Option B:** Use earthaccess to stream each day's granule from NASA Earthdata into memory as it is processed, saving you from having to store each day's worth of data.

The provided helper functions create a gridded netCDF for the `Daily_SIF_757nm` variable, but you can add to the list of variables if you wish. Note that the data is filtered to use exclusively samples with a `Quality_Flag` of 0, representing the best quality data. You may wish to include points with a `Quality_Flag` of 1 (good).


If you'd like to customize the gridded raster beyond what is shown in this example, here are some options you have:
- Change the start and end dates specified by the first two arguments to adjust the month referenced for the output product.
- Add additional variables to average in the output netCDF (ex: `Daily_SIF_771nm`, `Science_SIF_Relative_757nm`, `Science_daily_correction_factor`, etc.)
- Change the name of the output file by changing the value of the `daily_avg_file` variable.
- You can also specify a bounding box for the data, such as `lon_min=-130, lat_max=-65, lat_min=22, lat_max=50,` for the CONUS. An example of how to do this is commented out in the code blocks and just needs to be uncommented if you'd like to test it out.

In [ ]:
# Run this cell regardless of which option you choose
year = 2020
month = 6
start_date = datetime(year, month, 1)
_, num_days = calendar.monthrange(year, month)
end_date = datetime(year, month, num_days)

daily_avg_file = f"data/{dataset}/{start_date.strftime('%b_%Y').lower()}_sif.nc4"

**Option A:** Use the files you downloaded in Step III to create the gridded raster. This will take about 30 seconds now that you already have the data locally.

In [ ]:
# For this option, note that local_dataset has the value that is
# expected in the filename of your downloaded granules, rather than
# the name of the dataset on the DAAC.
local_dataset = "oco3_LtSIF"
create_gridded_raster(
    start_date,
    end_date,
    local_dataset,
    ["Daily_SIF_757nm"],
    daily_avg_file,
    lat_res = 0.5,
    lon_res = 0.5,
    # Uncomment to bound the data to the CONUS
    # lon_min=-150, lon_max=-60, lat_min=22, lat_max=50,
    local_dir=f"data/{dataset}/{year}-{month:02d}",
    filters={
        # Change Quality_Flag to ("<" 2) if you would like to use both best and good quality data
        "Quality_Flag": ("=", 0),
        # Filter to Nadir mode observations
        # "Metadata/MeasurementMode": ("=", 0)
    }
)

**Option B: Use earthaccess to stream data on the fly. This option uses less disk space but takes longer to finish processing, because each granule is transferred while the raster is being made.**

You can skip this block and proceed to plotting if you have already generated the `daily_avg_file` referenced in the block above.

In [ ]:
create_gridded_raster(
    start_date,
    end_date,
    dataset,
    ["Daily_SIF_757nm"],
    daily_avg_file,
    lat_res = 0.5,
    lon_res = 0.5,
    # Uncomment to bound the data to the CONUS
    lon_min=-130, lon_max=-65, lat_min=22, lat_max=50,
    filters={
        # Change Quality_Flag to (">" 2) if you would like to use both best and good quality data
        "Quality_Flag": ("=", 0)
    }
)

### Display the Monthly Gridded Raster

If you used a different dataset or variable for your monthly gridded raster, be sure to modify the values of `title`, `label`, and `outfile` in the code block below.

In [ ]:
ds = Dataset(daily_avg_file, "r")
var_name = "Daily_SIF_757nm"
var = ds[var_name]
lats = ds["lat"][:]
lons = ds["lon"][:]
fill_val = var._FillValue

daily_avg_data = var[...]
ds.close()
lon2d, lat2d = np.meshgrid(lons, lats)
# Transpose the meshgrid result to be of shape (lon_res, lat_res) ex. (360, 180)
lon2d = lon2d.T
lat2d = lat2d.T

# Create a masked array where the fill_val is masked
data_masked = np.ma.masked_where(daily_avg_data == fill_val, daily_avg_data)
# Average over axis 0 (the "time" dimension), produces a masked array of shape (lon_res, lat_res)
mean_data_masked = data_masked.mean(axis=0)

# Be sure to change the title and label if you change the monthly gridded raster you want to display
# To avoid overwriting the output image when making changes, you should also change the value of outfile
plot_gridded(
    mean_data_masked, lat2d, lon2d,
    vmax=0.8,
    vmin=0,
    cmap="YlGn",
    # Uncomment to window the plot to the CONUS
    # extents=[-130, -65, 22, 50],
    title=f"OCO-3 {start_date.strftime('%B %Y')} Mean Daily SIF$_{{757}}$",
    label=r"SIF (W/$\mathrm{m}^2$/sr/μm)",
    outfile=f"mean_oco3_{var_name.lower()}_{start_date.strftime('%B_%Y').lower()}.png"
)

## V. Create an OCO-2 SIF Monthly Gridded Raster

Next, we will repeat the process of generating and plotting a monthly gridded raster in an abridged manner for an OCO-2 SIF dataset. The process is largely the same, we are just changing the arguments to the functions and labels of plots. This section can serve as a template if you would like to create other gridded rasters, such as for an XCO<sub>2</sub> dataset.

In [ ]:
# These are all the dataset-dependent variables

# The dataset short name and version, see the Earthdata Search link in Section I if you want to change it.
# Version 11r covers September 2014 - March 2024, use version 11.2r for more recent OCO-2 data.
short_name = "OCO2_L2_Lite_SIF"
version = "11r"
dataset = f"{short_name}.{version}"
# This is the first part of the granule filename
local_dataset = "oco2_LtSIF"

mission_name = "OCO-2"

year = 2020
month = 6

# The name of the variable to use in the granule when making the gridded raster
var_name = "Daily_SIF_757nm"
# The name of the variable to use in the plot
var_label = "Daily SIF$_{{757}}$"
# Unit of the quantity being measured, this is also used in the plot
unit = r"W/$\mathrm{m}^2$/sr/μm"

start_date = datetime(year, month, 1)
_, num_days = calendar.monthrange(year, month)
end_date = datetime(year, month, num_days)

granule_dir = f"data/{dataset}/{year}-{month:02d}"
daily_avg_file = f"data/{dataset}/{start_date.strftime('%b_%Y').lower()}_{var_name}.nc4"

Download the granules, then generate the gridded raster.

In [ ]:
results = earthaccess.search_data(
    short_name=short_name,
    version=version,
    temporal=(start_date.date(), end_date.date(), True),
)
dl_granules = earthaccess.download(results, local_path=granule_dir)
print(f"Download summary: Downloaded {len(dl_granules)} out of {num_days} days")

create_gridded_raster(
    start_date,
    end_date,
    local_dataset,
    [var_name],
    daily_avg_file,
    lat_res = 0.5,
    lon_res = 0.5,
    # Uncomment to bound the data to the CONUS
    # lon_min=-130, lon_max=-65, lat_min=22, lat_max=50,
    local_dir=granule_dir,
)

In [ ]:
ds = Dataset(daily_avg_file, "r")
var = ds[var_name]
lats = ds["lat"][:]
lons = ds["lon"][:]
fill_val = var._FillValue

daily_avg_data = var[...]
ds.close()
lon2d, lat2d = np.meshgrid(lons, lats)
# Transpose the meshgrid result to be of shape (lon_res, lat_res) ex. (360, 180)
lon2d = lon2d.T
lat2d = lat2d.T

# Create a masked array where the fill_val is masked
data_masked = np.ma.masked_where(daily_avg_data == fill_val, daily_avg_data)
# Average over axis 0 (the "time" dimension), produces a masked array of shape (lon_res, lat_res)
mean_data_masked = data_masked.mean(axis=0)

# Be sure to change the title and label if you change the monthly gridded raster you want to display
# To avoid overwriting the output image when making changes, you should also change the value of outfile
plot_gridded(
    mean_data_masked, lat2d, lon2d,
    vmin=0.0,
    vmax=0.8,
    cmap="YlGn",
    # Uncomment to window the plot to the CONUS
    # extents=[-130, -65, 22, 50],
    title=f"{mission_name} {start_date.strftime('%B %Y')} Mean Daily {var_label}",
    label=f"{var_label} ({unit})",
    outfile=f"mean_{mission_name.lower()}_{var_name.lower()}_{start_date.strftime('%B_%Y').lower()}.png"
)

### Differences between OCO-2 and OCO-3 SIF products
While OCO-2 and OCO-3 use the same spectrometer design, a key difference we can see with between the gridded rasters that we produced is the pattern produced by their orbits. OCO-2 is mounted on its own spacecraft, and operates in a sun-synchronous orbit. In practice, this means that it will observe the same local solar time for a given latitude on any of its orbits. At the equator, OCO-2 provides observations at a local time of 1:15pm. 

On the other hand, the crossing orbit pattern that you can see in the OCO-3 gridded raster arises due to the lower orbital inclination angle of the ISS (OCO-2's inclination is close to perpendicular with the equatorial plane). One consequence of the ISS's orbit is that OCO-3 can observe the same area of the Earth at different times of day through successive revisits. The local time of OCO-3 SIF samples is included in the granules, but using it for analysis is not covered in this training exercise. Such information can be leveraged to train diurnal estimations of SIF, such as the GEOSIF model described in [Jeong et al., 2024](https://doi.org/10.1016/j.rse.2024.114284)